# ADRs Warehouse - Functionality Test Notebook

This notebook tests all the main functionality of the adrs_warehouse project:
1. **Configuration** - Verify ticker metadata and settings
2. **Data Fetching** - Download ADR data from Yahoo Finance
3. **Data Transformation** - Clean, normalize, and build dimensions
4. **Database Operations** - Create schema, load data, and query

In [45]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import duckdb
from datetime import datetime

## 1. Configuration Module

In [46]:
from adrs_warehouse.config import AR_ADRS, START_DATE, TICKER_METADATA

print(f"Number of ADRs configured: {len(AR_ADRS)}")
print(f"Tickers: {AR_ADRS}")
print(f"\nDefault start date: {START_DATE}")

Number of ADRs configured: 13
Tickers: ['YPF', 'GGAL', 'BMA', 'BBAR', 'PAM', 'TEO', 'CEPU', 'LOMA', 'CRESY', 'IRS', 'SUPV', 'MELI', 'BIOX']

Default start date: 2018-01-01


In [47]:
# Display ticker metadata as DataFrame
metadata_df = pd.DataFrame(TICKER_METADATA).T
metadata_df.index.name = 'ticker'
metadata_df

,company_name,exchange,sector,country
ticker,,,,
YPF,YPF S.A.,NYSE,Energy,Argentina
GGAL,Grupo Financiero Galicia S.A.,NASDAQ,Financial Services,Argentina
BMA,Banco Macro S.A.,NYSE,Financial Services,Argentina
BBAR,BBVA Argentina S.A.,NYSE,Financial Services,Argentina
PAM,Pampa Energia S.A.,NYSE,Utilities,Argentina
TEO,Telecom Argentina S.A.,NYSE,Communication Services,Argentina
CEPU,Central Puerto S.A.,NYSE,Utilities,Argentina
LOMA,Loma Negra Compania Industrial Argentina S.A.,NYSE,Materials,Argentina
CRESY,Cresud S.A.C.I.F.y A.,NASDAQ,Real Estate,Argentina


## 2. Data Fetching Module

In [48]:
from adrs_warehouse.data.fetch import download_adr_data, build_ticker_dimension

# Test with a subset of tickers and recent date range for faster execution
test_tickers = ['YPF', 'MELI', 'GGAL']
test_start = '2024-01-01'

print(f"Downloading data for {test_tickers} from {test_start}...")
raw_data = download_adr_data(tickers=test_tickers, start_date=test_start)
print(f"\nShape: {raw_data.shape}")
raw_data.head()

[*********************100%***********************]  3 of 3 completed

Download complete. Shape: (517, 15)

Shape: (517, 15)


Ticker             MELI                                                 \
Price              Open         High          Low        Close  Volume   
Date                                                                     
2024-01-02  1562.609985  1562.609985  1518.119995  1529.160034  350200   
2024-01-03  1515.010010  1523.189941  1497.900024  1500.000000  272400   
2024-01-04  1489.520020  1543.069946  1483.640015  1519.380005  436400   
2024-01-05  1527.079956  1559.660034  1527.079956  1538.829956  317400   
2024-01-08  1548.180054  1579.380005  1548.180054  1575.599976  278600   

Ticker           GGAL                                                 YPF  \
Price            Open       High        Low      Close  Volume       Open   
Date                                                                        
2024-01-02  15.540088  15.685069  15.141392  15.367924  984200  17.150000   
2024-01-03  15.259192  15.594458  15.168579  15.204824  673700  16.600000   
2024-01-04  15.277314  15.449477  14.661147  14.715514  921600  16.709999   
2024-01-05  14.588657  15.077965  14.588657  14.942046  865900  16.379999   
2024-01-08  15.077965  15.077965  14.588657  15.041720  469600  16.549999   

Ticker                                                
Price            High        Low      Close   Volume  
Date                                                  
2024-01-02  17.250000  16.469999  16.520000  2197800  
2024-01-03  17.070000  16.559999  16.770000  2152300  
2024-01-04  16.870001  16.219999  16.230000  1627700  
2024-01-05  16.700001  16.230000  16.700001  1788800  
2024-01-08  16.719999  16.219999  16.709999  1181700

In [49]:
# Check the column structure (MultiIndex)
print("Column levels:")
print(f"  Level 0 (Ticker): {raw_data.columns.get_level_values(0).unique().tolist()}")
print(f"  Level 1 (Price): {raw_data.columns.get_level_values(1).unique().tolist()}")

Column levels:
  Level 0 (Ticker): ['MELI', 'GGAL', 'YPF']
  Level 1 (Price): ['Open', 'High', 'Low', 'Close', 'Volume']


In [50]:
# Test build_ticker_dimension from fetch module
ticker_info = build_ticker_dimension(raw_data)
print("Ticker dimension from fetch module:")
ticker_info

Ticker dimension from fetch module:


,ticker,has_data,first_date,last_date
1,GGAL,True,2024-01-02,2026-01-23
0,MELI,True,2024-01-02,2026-01-23
2,YPF,True,2024-01-02,2026-01-23


## 3. Data Transformation Module

In [51]:
from adrs_warehouse.data.transform import (
    clean_data,
    normalize_prices_long,
    build_date_dimension,
    build_ticker_dimension as build_ticker_dim,
    build_fact_table
)

### 3.1 Clean Data

In [52]:
# Clean the raw data (removes columns with all nulls)
cleaned_data = clean_data(raw_data)
print(f"Shape before cleaning: {raw_data.shape}")
print(f"Shape after cleaning: {cleaned_data.shape}")
cleaned_data.head()

Shape before cleaning: (517, 15)
Shape after cleaning: (517, 15)


Ticker             MELI                                                   \
Price              Open         High          Low        Close    Volume   
Date                                                                       
2024-01-02  1562.609985  1562.609985  1518.119995  1529.160034  350200.0   
2024-01-03  1515.010010  1523.189941  1497.900024  1500.000000  272400.0   
2024-01-04  1489.520020  1543.069946  1483.640015  1519.380005  436400.0   
2024-01-05  1527.079956  1559.660034  1527.079956  1538.829956  317400.0   
2024-01-08  1548.180054  1579.380005  1548.180054  1575.599976  278600.0   

Ticker           GGAL                                                   YPF  \
Price            Open       High        Low      Close    Volume       Open   
Date                                                                          
2024-01-02  15.540088  15.685069  15.141392  15.367924  984200.0  17.150000   
2024-01-03  15.259192  15.594458  15.168579  15.204824  673700.0  16.600000   
2024-01-04  15.277314  15.449477  14.661147  14.715514  921600.0  16.709999   
2024-01-05  14.588657  15.077965  14.588657  14.942046  865900.0  16.379999   
2024-01-08  15.077965  15.077965  14.588657  15.041720  469600.0  16.549999   

Ticker                                                  
Price            High        Low      Close     Volume  
Date                                                    
2024-01-02  17.250000  16.469999  16.520000  2197800.0  
2024-01-03  17.070000  16.559999  16.770000  2152300.0  
2024-01-04  16.870001  16.219999  16.230000  1627700.0  
2024-01-05  16.700001  16.230000  16.700001  1788800.0  
2024-01-08  16.719999  16.219999  16.709999  1181700.0

### 3.2 Normalize to Long Format

In [53]:
# Convert MultiIndex to long/tidy format
long_data = normalize_prices_long(cleaned_data)
print(f"Long format shape: {long_data.shape}")
print(f"Columns: {long_data.columns.tolist()}")
long_data.head(10)

Long format shape: (1551, 7)
Columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']


,date,ticker,open,high,low,close,volume
0,2024-01-02,MELI,1562.609985,1562.609985,1518.119995,1529.160034,350200.0
1,2024-01-02,GGAL,15.540088,15.685069,15.141392,15.367924,984200.0
2,2024-01-02,YPF,17.150000,17.250000,16.469999,16.520000,2197800.0
3,2024-01-03,MELI,1515.010010,1523.189941,1497.900024,1500.000000,272400.0
4,2024-01-03,GGAL,15.259192,15.594458,15.168579,15.204824,673700.0
5,2024-01-03,YPF,16.600000,17.070000,16.559999,16.770000,2152300.0
6,2024-01-04,MELI,1489.520020,1543.069946,1483.640015,1519.380005,436400.0
7,2024-01-04,GGAL,15.277314,15.449477,14.661147,14.715514,921600.0
8,2024-01-04,YPF,16.709999,16.870001,16.219999,16.230000,1627700.0
9,2024-01-05,MELI,1527.079956,1559.660034,1527.079956,1538.829956,317400.0


In [54]:
# Verify unique tickers and date range
print(f"Unique tickers: {long_data['ticker'].unique().tolist()}")
print(f"Date range: {long_data['date'].min()} to {long_data['date'].max()}")
print(f"Records per ticker:")
long_data.groupby('ticker').size()

Unique tickers: ['MELI', 'GGAL', 'YPF']
Date range: 2024-01-02 00:00:00 to 2026-01-23 00:00:00
Records per ticker:


ticker
GGAL    517
MELI    517
YPF     517
dtype: int64

### 3.3 Build Date Dimension

In [55]:
# Build date dimension from the long data
dim_date = build_date_dimension(cleaned_data)
print(f"Date dimension shape: {dim_date.shape}")
print(f"Columns: {dim_date.columns.tolist()}")
dim_date.head(10)

Date dimension shape: (517, 13)
Columns: ['date', 'date_id', 'year', 'quarter', 'month', 'month_name', 'day', 'day_of_week', 'day_name', 'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end']


,date,date_id,year,quarter,month,month_name,day,day_of_week,day_name,week_of_year,is_weekend,is_month_start,is_month_end
0,2024-01-02,20240102,2024,1,1,January,2,1,Tuesday,1,False,False,False
1,2024-01-03,20240103,2024,1,1,January,3,2,Wednesday,1,False,False,False
2,2024-01-04,20240104,2024,1,1,January,4,3,Thursday,1,False,False,False
3,2024-01-05,20240105,2024,1,1,January,5,4,Friday,1,False,False,False
4,2024-01-08,20240108,2024,1,1,January,8,0,Monday,2,False,False,False
5,2024-01-09,20240109,2024,1,1,January,9,1,Tuesday,2,False,False,False
6,2024-01-10,20240110,2024,1,1,January,10,2,Wednesday,2,False,False,False
7,2024-01-11,20240111,2024,1,1,January,11,3,Thursday,2,False,False,False
8,2024-01-12,20240112,2024,1,1,January,12,4,Friday,2,False,False,False
9,2024-01-16,20240116,2024,1,1,January,16,1,Tuesday,3,False,False,False


In [56]:
# Verify date dimension attributes
print("Sample of date dimension attributes:")
dim_date.sample(5)

Sample of date dimension attributes:


,date,date_id,year,quarter,month,month_name,day,day_of_week,day_name,week_of_year,is_weekend,is_month_start,is_month_end
280,2025-02-13,20250213,2025,1,2,February,13,3,Thursday,7,False,False,False
46,2024-03-08,20240308,2024,1,3,March,8,4,Friday,10,False,False,False
177,2024-09-16,20240916,2024,3,9,September,16,0,Monday,38,False,False,False
495,2025-12-22,20251222,2025,4,12,December,22,0,Monday,52,False,False,False
437,2025-09-30,20250930,2025,3,9,September,30,1,Tuesday,40,False,False,True


### 3.4 Build Ticker Dimension

In [57]:
# Build ticker dimension with metadata
dim_ticker = build_ticker_dim(cleaned_data, TICKER_METADATA)
print(f"Ticker dimension shape: {dim_ticker.shape}")
print(f"Columns: {dim_ticker.columns.tolist()}")
dim_ticker

Ticker dimension shape: (3, 8)
Columns: ['ticker_id', 'ticker_symbol', 'company_name', 'exchange', 'sector', 'country', 'first_trade_date', 'last_trade_date']


,ticker_id,ticker_symbol,company_name,exchange,sector,country,first_trade_date,last_trade_date
0,1,GGAL,Grupo Financiero Galicia S.A.,NASDAQ,Financial Services,Argentina,2024-01-02,2026-01-23
1,2,MELI,"MercadoLibre, Inc.",NASDAQ,Consumer Cyclical,Argentina,2024-01-02,2026-01-23
2,3,YPF,YPF S.A.,NYSE,Energy,Argentina,2024-01-02,2026-01-23


### 3.5 Build Fact Table

In [58]:
# Build fact table with foreign keys to dimensions
fact_prices = build_fact_table(cleaned_data, dim_date, dim_ticker)
print(f"Fact table shape: {fact_prices.shape}")
print(f"Columns: {fact_prices.columns.tolist()}")
fact_prices.head(10)

Fact table shape: (1551, 8)
Columns: ['date_id', 'ticker_id', 'open_price', 'high_price', 'low_price', 'close_price', 'adj_close_price', 'volume']


,date_id,ticker_id,open_price,high_price,low_price,close_price,adj_close_price,volume
0,20240102,2,1562.609985,1562.609985,1518.119995,1529.160034,1529.160034,350200.0
1,20240102,1,15.540088,15.685069,15.141392,15.367924,15.367924,984200.0
2,20240102,3,17.150000,17.250000,16.469999,16.520000,16.520000,2197800.0
3,20240103,2,1515.010010,1523.189941,1497.900024,1500.000000,1500.000000,272400.0
4,20240103,1,15.259192,15.594458,15.168579,15.204824,15.204824,673700.0
5,20240103,3,16.600000,17.070000,16.559999,16.770000,16.770000,2152300.0
6,20240104,2,1489.520020,1543.069946,1483.640015,1519.380005,1519.380005,436400.0
7,20240104,1,15.277314,15.449477,14.661147,14.715514,14.715514,921600.0
8,20240104,3,16.709999,16.870001,16.219999,16.230000,16.230000,1627700.0
9,20240105,2,1527.079956,1559.660034,1527.079956,1538.829956,1538.829956,317400.0


In [59]:
# Verify fact table data types and statistics
print("Fact table info:")
fact_prices.info()
print("\nFact table statistics:")
fact_prices.describe()

Fact table info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1551 entries, 0 to 1550
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date_id          1551 non-null   int64  
 1   ticker_id        1551 non-null   int64  
 2   open_price       1551 non-null   float64
 3   high_price       1551 non-null   float64
 4   low_price        1551 non-null   float64
 5   close_price      1551 non-null   float64
 6   adj_close_price  1551 non-null   float64
 7   volume           1551 non-null   float64
dtypes: float64(6), int64(2)
memory usage: 97.1 KB

Fact table statistics:


,date_id,ticker_id,open_price,high_price,low_price,close_price,adj_close_price,volume
count,1.551000e+03,1551.00000,1551.000000,1551.000000,1551.000000,1551.000000,1551.000000,1.551000e+03
mean,2.024607e+07,2.00000,687.158368,696.843149,676.282920,686.713290,686.713290,1.325357e+06
std,5.506766e+03,0.81676,938.418305,951.040952,923.813898,937.745612,937.745612,1.276543e+06
min,2.024010e+07,1.00000,13.954367,14.498042,13.881877,14.035918,14.035918,9.820000e+04
25%,2.024071e+07,1.00000,29.604882,30.297801,28.562450,29.388905,29.388905,4.361000e+05
50%,2.025011e+07,2.00000,47.419998,48.654344,46.037050,47.496380,47.496380,9.890000e+05
75%,2.025072e+07,3.00000,1728.664978,1749.659973,1707.155029,1732.794983,1732.794983,1.746500e+06
max,2.026012e+07,3.00000,2645.219971,2645.219971,2582.000000,2613.629883,2613.629883,1.563180e+07


## 4. Database Operations Module

In [60]:
from adrs_warehouse.database.operations import ADRDatabase
from adrs_warehouse.database.schema import ALL_DDL

### 4.1 Create In-Memory Database

In [61]:
# Create an in-memory database for testing
db = ADRDatabase(':memory:')
print("Database connection established (in-memory)")

Database connection established (in-memory)


### 4.2 Create Star Schema

In [62]:
# Create the star schema
db.create_star_schema()
print("Star schema created successfully")

Star schema created successfully


In [63]:
# Verify schema was created
tables = db.query("SHOW TABLES")
print("Tables in database:")
tables

Tables in database:


,name
0,dim_date
1,dim_ticker
2,fact_stock_prices


### 4.3 Load Dimensions

In [64]:
# Load date dimension
db.load_dimension(dim_date, 'dim_date')
print(f"Loaded {len(dim_date)} rows into dim_date")

# Verify
db.query("SELECT COUNT(*) as count FROM dim_date")

Loaded 517 rows into dim_date


,count
0,517


In [65]:
# Load ticker dimension
db.load_dimension(dim_ticker, 'dim_ticker')
print(f"Loaded {len(dim_ticker)} rows into dim_ticker")

# Verify
db.query("SELECT * FROM dim_ticker")

Loaded 3 rows into dim_ticker


,ticker_id,ticker_symbol,company_name,exchange,sector,country,first_trade_date,last_trade_date
0,1,GGAL,Grupo Financiero Galicia S.A.,NASDAQ,Financial Services,Argentina,2024-01-02,2026-01-23
1,2,MELI,"MercadoLibre, Inc.",NASDAQ,Consumer Cyclical,Argentina,2024-01-02,2026-01-23
2,3,YPF,YPF S.A.,NYSE,Energy,Argentina,2024-01-02,2026-01-23


### 4.4 Load Fact Table

In [66]:
# Load fact table
db.load_fact(fact_prices)
print(f"Loaded {len(fact_prices)} rows into fact_stock_prices")

# Verify
db.query("SELECT COUNT(*) as count FROM fact_stock_prices")

Loaded 1551 rows into fact_stock_prices


,count
0,1551


### 4.5 Query the Star Schema

In [67]:
# Get schema info
schema_info = db.get_schema_info()
print("Schema Information:")
for table, info in schema_info.items():
    print(f"\n{table}: {info['row_count']} rows")
    print(f"  Columns: {info['columns']}")

Schema Information:

dim_date: 517 rows
  Columns: [{'column_name': 'date_id', 'column_type': 'INTEGER', 'null': 'NO', 'key': 'PRI', 'default': None, 'extra': None}, {'column_name': 'date', 'column_type': 'DATE', 'null': 'NO', 'key': None, 'default': None, 'extra': None}, {'column_name': 'year', 'column_type': 'INTEGER', 'null': 'NO', 'key': None, 'default': None, 'extra': None}, {'column_name': 'quarter', 'column_type': 'INTEGER', 'null': 'NO', 'key': None, 'default': None, 'extra': None}, {'column_name': 'month', 'column_type': 'INTEGER', 'null': 'NO', 'key': None, 'default': None, 'extra': None}, {'column_name': 'month_name', 'column_type': 'VARCHAR', 'null': 'NO', 'key': None, 'default': None, 'extra': None}, {'column_name': 'day', 'column_type': 'INTEGER', 'null': 'NO', 'key': None, 'default': None, 'extra': None}, {'column_name': 'day_of_week', 'column_type': 'INTEGER', 'null': 'NO', 'key': None, 'default': None, 'extra': None}, {'column_name': 'day_name', 'column_type': 'VARCHAR

In [68]:
# Test analytical query: Average close price by ticker
query = """
SELECT 
    t.ticker_symbol AS ticker,
    t.company_name,
    COUNT(*) as trading_days,
    ROUND(AVG(f.close_price), 2) as avg_close,
    ROUND(MIN(f.close_price), 2) as min_close,
    ROUND(MAX(f.close_price), 2) as max_close
FROM fact_stock_prices f
JOIN dim_ticker t ON f.ticker_id = t.ticker_id
GROUP BY t.ticker_symbol, t.company_name
ORDER BY avg_close DESC
"""
db.query(query)

,ticker,company_name,trading_days,avg_close,min_close,max_close
0,MELI,"MercadoLibre, Inc.",517,1988.52,1356.43,2613.63
1,GGAL,Grupo Financiero Galicia S.A.,517,42.89,14.04,70.53
2,YPF,YPF S.A.,517,28.73,14.91,46.03


In [69]:
query = """
SELECT *
FROM fact_stock_prices
"""

db.query(query)

,date_id,ticker_id,open_price,high_price,low_price,close_price,adj_close_price,volume
0,20240102,2,1562.609985,1562.609985,1518.119995,1529.160034,1529.160034,350200
1,20240102,1,15.540088,15.685069,15.141392,15.367924,15.367924,984200
2,20240102,3,17.150000,17.250000,16.469999,16.520000,16.520000,2197800
3,20240103,2,1515.010010,1523.189941,1497.900024,1500.000000,1500.000000,272400
4,20240103,1,15.259192,15.594458,15.168579,15.204824,15.204824,673700
...,...,...,...,...,...,...,...,...
1546,20260122,1,54.250000,55.400002,53.490002,53.770000,53.770000,810000
1547,20260122,3,36.000000,36.299999,35.160000,35.689999,35.689999,1076400
1548,20260123,2,2145.000000,2153.459961,2100.310059,2137.290039,2137.290039,467100
1549,20260123,1,54.200001,54.680000,53.369999,53.740002,53.740002,898100


In [70]:
# Test analytical query: Monthly average prices
query = """
SELECT 
    d.year,
    d.month,
    t.ticker_symbol,
    ROUND(AVG(f.close_price), 2) as avg_close,
    SUM(f.volume) as total_volume
FROM fact_stock_prices f
JOIN dim_date d ON f.date_id = d.date_id
JOIN dim_ticker t ON f.ticker_id = t.ticker_id
GROUP BY t.ticker_symbol, d.year, d.month
ORDER BY t.ticker_symbol, d.year, d.month
LIMIT 15
"""
db.query(query)

,year,month,ticker_symbol,avg_close,total_volume
0,2024,1,GGAL,16.13,24315000.0
1,2024,2,GGAL,18.75,18090300.0
2,2024,3,GGAL,21.72,16974100.0
3,2024,4,GGAL,25.89,25615800.0
4,2024,5,GGAL,31.64,23782900.0
5,2024,6,GGAL,30.27,20765800.0
6,2024,7,GGAL,26.58,17057700.0
7,2024,8,GGAL,32.14,26100300.0
8,2024,9,GGAL,42.61,21658700.0
9,2024,10,GGAL,47.56,20221000.0


In [71]:
# Test analytical query: Day of week analysis
query = """
SELECT 
    d.day_name,
    COUNT(*) as trading_days,
    ROUND(AVG(f.volume), 0) as avg_volume
FROM fact_stock_prices f
JOIN dim_date d ON f.date_id = d.date_id
WHERE d.is_weekend = FALSE
GROUP BY d.day_name
ORDER BY 
    CASE d.day_name 
        WHEN 'Monday' THEN 1
        WHEN 'Tuesday' THEN 2
        WHEN 'Wednesday' THEN 3
        WHEN 'Thursday' THEN 4
        WHEN 'Friday' THEN 5
    END
"""
db.query(query)

,day_name,trading_days,avg_volume
0,Monday,294,1445542.0
1,Tuesday,324,1278967.0
2,Wednesday,315,1315312.0
3,Thursday,303,1317092.0
4,Friday,315,1278893.0


### 4.6 Cleanup

In [72]:
# Close the database connection
db.close()
print("Database connection closed")

Database connection closed


## 5. Full Pipeline Test (All Tickers)

In [73]:
# Uncomment to run full pipeline with all tickers (takes longer)
# This downloads data for all 13 ADRs since 2018

# print("Downloading all ADR data...")
# full_data = download_adr_data()  # Uses defaults from config
# print(f"Raw data shape: {full_data.shape}")

# print("\nTransforming data...")
# full_cleaned = clean_data(full_data)
# full_long = normalize_prices_long(full_cleaned)
# full_dim_date = build_date_dimension(full_long)
# full_dim_ticker = build_ticker_dim(full_long, TICKER_METADATA)
# full_fact = build_fact_table(full_long, full_dim_date, full_dim_ticker)

# print(f"\nFull pipeline results:")
# print(f"  Long data: {full_long.shape}")
# print(f"  Date dimension: {full_dim_date.shape}")
# print(f"  Ticker dimension: {full_dim_ticker.shape}")
# print(f"  Fact table: {full_fact.shape}")

## Summary

All tests completed successfully! The adrs_warehouse project correctly:

1. **Fetches data** from Yahoo Finance for Argentine ADRs
2. **Cleans** the raw data by removing null columns
3. **Normalizes** MultiIndex data to long/tidy format
4. **Builds dimensions** (date and ticker) with proper attributes
5. **Creates a star schema** in DuckDB with proper DDL
6. **Loads data** into the warehouse (dimensions and fact table)
7. **Supports analytical queries** joining fact and dimension tables